<a href="https://colab.research.google.com/github/Satdev-Singh/TestRepo/blob/main/ETL_COMPLETO_Reducci%C3%B3n_de_Fricci%C3%B3n_de_Datos_en_Marketing_Digital.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TFE: Reducción de fricción de datos mediante Business Intelligence
#### Autor  : Fernando López Reyes
#### Máster : Business Intelligence — Universidad Internacional de La Rioja

> Pipeline ETL que extrae, transforma y carga cinco fuentes de datos
de marketing digital, resolviendo fricción técnica, estructural y
semántica para producir un modelo dimensional Kimball


In [ ]:
from google.colab import drive
drive.mount ('/content/drive')

import pandas as pd
import numpy as np
import unicodedata
from datetime import datetime
import warnings
import os
warnings.filterwarnings('ignore')


Mounted at /content/drive


In [ ]:
# CONFIGURACIÓN DE RUTAS

meta_ads = pd.read_excel("/content/drive/MyDrive/Tesis/Dataset_Marketing_Digital_FRICTION.xlsx", sheet_name = "Meta_Ads_RAW")
google_ads = pd.read_excel("/content/drive/MyDrive/Tesis/Dataset_Marketing_Digital_FRICTION.xlsx", sheet_name = "Google_Ads_RAW")
mail_mkt = pd.read_excel("/content/drive/MyDrive/Tesis/Dataset_Marketing_Digital_FRICTION.xlsx", sheet_name = "Email_Marketing_RAW")
ppto_cmp = pd.read_excel("/content/drive/MyDrive/Tesis/Dataset_Marketing_Digital_FRICTION.xlsx", sheet_name = "Presupuesto_Campañas_RAW")
leads_cv = pd.read_excel("/content/drive/MyDrive/Tesis/Dataset_Marketing_Digital_FRICTION.xlsx", sheet_name = "Leads_Conversiones_RAW")

print("meta_ads ha sido cargada con éxito")
print("google_ads ha sido cargada con éxito")
print("mail_mkt ha sido cargada con éxito")
print("ppto_cmp ha sido cargada con éxito")
print("leads_cv ha sido cargada con éxito")

meta_ads ha sido cargada con éxito
google_ads ha sido cargada con éxito
mail_mkt ha sido cargada con éxito
ppto_cmp ha sido cargada con éxito
leads_cv ha sido cargada con éxito


In [ ]:
# CONFIGURACIÓN DE RUTAS

INPUT_FILE   = "/content/drive/MyDrive/Tesis/Dataset_Marketing_Digital_FRICTION.xlsx"
OUTPUT_MODEL = 'Modelo_Dimensional_Marketing.xlsx'
OUTPUT_LOG   = 'Log_Errores_ETL.csv'


# DICCIONARIOS CANÓNICOS
# Resuelve fricción semántica: 28 variantes → 15 nombres únicos
CAMPANA_MAP = {
    'Black Friday 2023': 'Black Friday 2023',
    'BlackFriday23':     'Black Friday 2023',
    'BF2023':            'Black Friday 2023',
    'Navidad 2023':      'Navidad 2023',
    'Navidad_2023':      'Navidad 2023',
    'NAVIDAD23':         'Navidad 2023',
    'Verano 2024':       'Verano 2024',
    'Verano_2024':       'Verano 2024',
    'Lanzamiento Producto A': 'Lanzamiento Producto A',
    'Lanzamiento Prod. A':    'Lanzamiento Producto A',
    'Lanzamiento A':          'Lanzamiento Producto A',
    'Rebranding Marzo':    'Rebranding Marzo',
    'Rebranding - Marzo':  'Rebranding Marzo',
    'Rebranding Marzo 24': 'Rebranding Marzo',
    'Día de la Madre':  'Día de la Madre',
    'Dia de la Madre':  'Día de la Madre',
    'Madres 2024':      'Día de la Madre',
    'CyberDay Mayo':      'CyberDay Mayo',
    'CyberDay Mayo 2024': 'CyberDay Mayo',
    'Cyber Day':          'CyberDay Mayo',
    'Back to School':    'Back to School',
    'Back to School 24': 'Back to School',
    'BTS Agosto':        'Back to School',
    'Campaña Awareness Q3':        'Awareness Q3',
    'Awareness Q3':                'Awareness Q3',
    'Campaña Awareness - Q3 2024': 'Awareness Q3',
    'Retargeting Agosto':        'Retargeting Agosto',
    'Retargeting AGO':           'Retargeting Agosto',
    'Retargeting - Agosto 2024': 'Retargeting Agosto',
    'Fidelización Clientes VIP': 'Fidelización VIP',
    'VIP Fidelización':          'Fidelización VIP',
    'Clientes VIP':              'Fidelización VIP',
    'Campaña Influencer Sep': 'Influencer Septiembre',
    'Influencer Septiembre':  'Influencer Septiembre',
    'Sep_Influencer':         'Influencer Septiembre',
    'Halloween Promo':      'Halloween',
    'Halloween Promo 2023': 'Halloween',
    'Halloween':            'Halloween',
    'Año Nuevo 2024':  'Año Nuevo 2024',
    'AñoNuevo2024':    'Año Nuevo 2024',
    'New Year 2024':   'Año Nuevo 2024',
}

# Resuelve fricción semántica: 11 variantes → 5 estados canónicos
ESTADO_MAP = {
    'ACTIVA': 'Activo', 'activa': 'Activo', 'Activa': 'Activo',
    'enabled': 'Activo', 'ACTIVO': 'Activo', 'Activo': 'Activo',
    'Enabled': 'Activo',
    'PAUSADA': 'Pausado', 'Pausada': 'Pausado', 'Paused': 'Pausado',
    'PAUSED': 'Pausado', 'pausada': 'Pausado',
    'finalizada': 'Finalizado', 'Finalizada': 'Finalizado',
    'FINALIZADA': 'Finalizado', 'Removed': 'Finalizado',
    'En revisión': 'En revisión', 'En Revision': 'En revisión',
    'En revision': 'En revisión',
}

# Resuelve fricción semántica en canales
CANAL_MAP = {
    'Meta Ads': 'Meta Ads', 'meta': 'Meta Ads', 'Meta': 'Meta Ads',
    'Meta Ads ': 'Meta Ads',
    'Google Ads': 'Google Ads', 'GOOGLE': 'Google Ads',
    'Google Ads ': 'Google Ads',
    'Email': 'Email Marketing', 'email': 'Email Marketing',
    'email marketing': 'Email Marketing', '  Email  ': 'Email Marketing',
    'TikTok Ads': 'TikTok Ads',
    'LinkedIn Ads': 'LinkedIn Ads',
    'Orgánico': 'Orgánico', 'organic': 'Orgánico',
    'Referido': 'Referido', 'referral': 'Referido',
}


# Resuelve fricción semántica en responsables (abreviaturas → nombre completo)
RESPONSABLE_MAP = {
    'C. Perez': 'Camila Perez', 'Camila Perez': 'Camila Perez', 'Camila Pérez': 'Camila Perez',
    'D. Torres': 'Diego Torres', 'Diego Torres': 'Diego Torres',
    'M. Gonzalez': 'Maria Gonzalez', 'Maria Gonzalez': 'Maria Gonzalez', 'Maria González': 'Maria Gonzalez',
    'R. Munoz': 'Rodrigo Munoz', 'Rodrigo Munoz': 'Rodrigo Munoz', 'Rodrigo Muñoz': 'Rodrigo Munoz',
    'Carlos Mendez': 'Carlos Mendez',
    'Ana Ruiz': 'Ana Ruiz',
    'Laura Vega': 'Laura Vega',
    'Pedro Soto': 'Pedro Soto',
}

# Roles asignados por responsable (atributo de Dim_Responsable)
RESPONSABLE_ROL = {
    'Camila Perez':  'Gestor de Pauta Senior',
    'Diego Torres':  'Gestor de Pauta',
    'Maria Gonzalez':'Coordinadora de Marketing',
    'Rodrigo Munoz': 'Gestor de Pauta',
    'Carlos Mendez': 'Analista de Campañas',
    'Ana Ruiz':      'Gestora de Pauta Senior',
    'Laura Vega':    'Analista de Campañas',
    'Pedro Soto':    'Ejecutivo de Email Marketing',
}

# Resuelve fricción semántica en monedas
MONEDA_MAP = {
    'USD': 'USD', 'usd': 'USD', '$': 'USD',
    'Dólares': 'USD', 'dolares': 'USD', 'CLP': 'CLP',
}

# Resuelve fricción semántica en estado de pago
PAGO_MAP = {
    'Pagado': 'Pagado', 'pagado': 'Pagado', 'PAGADO': 'Pagado',
    'Pendiente': 'Pendiente', 'pendiente': 'Pendiente', 'Por pagar': 'Pendiente',
    'En proceso': 'En proceso',
}

# Resuelve fricción semántica en estado de leads
LEAD_ESTADO_MAP = {
    'Nuevo': 'Nuevo', 'nuevo': 'Nuevo',
    'Contactado': 'Contactado', 'CONTACTADO': 'Contactado',
    'Calificado': 'Calificado', 'calificado': 'Calificado',
    'Cerrado Ganado': 'Ganado', 'ganado': 'Ganado',
    'Cerrado Perdido': 'Perdido', 'perdido': 'Perdido',
    'En proceso': 'En proceso',
}




In [ ]:
# FUNCIONES AUXILIARES

def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")

def normalizar_texto(texto):
    """Normaliza nombres: strip + quita tildes + Title Case."""
    if pd.isna(texto):
        return None
    t = str(texto).strip()
    t = unicodedata.normalize('NFKD', t)
    t = ''.join(c for c in t if not unicodedata.combining(c))
    return t.title()

def normalizar_responsable(texto):
    """Normaliza responsable: quita tildes/espacios + mapea abreviaturas a nombre completo."""
    if pd.isna(texto):
        return None
    base = normalizar_texto(texto)
    return RESPONSABLE_MAP.get(base, base)

def normalizar_fecha(serie):
    """Convierte múltiples formatos de fecha a ISO 8601."""
    return pd.to_datetime(serie, dayfirst=True, errors='coerce')

# Log de errores global
errores_log = []

def registrar_error(fuente, transformacion, registros_afectados, accion):
    errores_log.append({
        'fuente': fuente,
        'transformacion': transformacion,
        'registros_afectados': registros_afectados,
        'accion': accion,
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    })

In [ ]:
# EXTRACCIÓN
def extraer():
    log("═" * 60)
    log("FASE 1: EXTRACCIÓN")
    log("═" * 60)

    hojas = {
        'meta':   'Meta_Ads_RAW',
        'google': 'Google_Ads_RAW',
        'email':  'Email_Marketing_RAW',
        'pres':   'Presupuesto_Campañas_RAW',
        'leads':  'Leads_Conversiones_RAW',
    }

    dfs = {}
    for clave, hoja in hojas.items():
        df = pd.read_excel(INPUT_FILE, sheet_name=hoja, dtype=str)
        dfs[clave] = df
        log(f"  {hoja}: {len(df)} registros × {len(df.columns)} columnas")

    total = sum(len(df) for df in dfs.values())
    log(f"  TOTAL EXTRAÍDO: {total} registros de 5 fuentes")
    return dfs


# TRANSFORMACIÓN — META ADS

def transformar_meta(df_raw):
    log("\n── Transformando Meta Ads ──")
    df = df_raw.copy()
    src = 'Meta Ads'

    # T1. Normalización de fechas
    df['fecha_norm'] = normalizar_fecha(df['Fecha'])
    nulos_fecha = df['fecha_norm'].isna().sum()
    registrar_error(src, 'Normalización fechas', int(nulos_fecha), 'NaT por formato no parseable')
    log(f"  T1 Fechas: {nulos_fecha} no parseadas")

    # T2. Deduplicación
    antes = len(df)
    df = df.drop_duplicates(subset=['Nombre_Campaña', 'Fecha', 'Gasto_USD'], keep='first').reset_index(drop=True)
    dupl = antes - len(df)
    registrar_error(src, 'Deduplicación', dupl, 'Registros eliminados')
    log(f"  T2 Duplicados: {dupl} eliminados ({dupl/antes*100:.1f}%)")

    # T3. Mapeo canónico de campañas
    df['campana_canon'] = df['Nombre_Campaña'].map(CAMPANA_MAP).fillna(df['Nombre_Campaña'])
    no_map = (df['campana_canon'] == df['Nombre_Campaña']).sum()
    registrar_error(src, 'Mapeo campañas', int(no_map), 'Sin mapeo (nombre original conservado)')
    log(f"  T3 Campañas: {df['Nombre_Campaña'].nunique()} variantes → {df['campana_canon'].nunique()} canónicas")

    # T4. Normalización de estados
    df['estado_norm'] = df['Estado'].map(ESTADO_MAP).fillna('Sin clasificar')
    log(f"  T4 Estados: {df['Estado'].nunique()} variantes → {df['estado_norm'].nunique()} canónicos")

    # T5. Normalización de responsables
    df['responsable_norm'] = df['Responsable'].apply(normalizar_responsable)
    log(f"  T5 Responsables: {df['Responsable'].nunique()} variantes → {df['responsable_norm'].nunique()} canónicos")

    # T6. Conversión numérica
    cols_num = ['Impresiones', 'Clics', 'CTR_%', 'Gasto_USD', 'Conversiones', 'CPA_USD', 'ROAS']
    for col in cols_num:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # T7. Imputación de nulos con mediana por campaña
    nulos_gasto = int(df['Gasto_USD'].isna().sum())
    df['Gasto_USD'] = df.groupby('campana_canon')['Gasto_USD'].transform(lambda x: x.fillna(x.median()))
    imputados = nulos_gasto - int(df['Gasto_USD'].isna().sum())
    registrar_error(src, 'Imputación nulos gasto', imputados, 'Mediana por campaña aplicada')
    log(f"  T7 Nulos gasto: {nulos_gasto} detectados → {imputados} imputados con mediana")

    df['fuente'] = 'Meta Ads'
    log(f"  Resultado: {len(df)} registros limpios")
    return df


# TRANSFORMACIÓN — GOOGLE ADS

def transformar_google(df_raw):
    log("\n── Transformando Google Ads ──")
    df = df_raw.copy()
    src = 'Google Ads'

    # T8. Homologación de columnas (fricción estructural)
    df = df.rename(columns={
        'Codigo_Registro':     'ID_Registro',
        'Fecha_Registro':      'Fecha',
        'Campaign_Name':       'Nombre_Campaña',
        'Tipo_Objetivo':       'Objetivo',
        'Grupo_Etario':        'Segmento_Edad',
        'Impresiones_Totales': 'Impresiones',
        'Clicks':              'Clics',
        'CTR':                 'CTR_%',
        'Inversion_Total':     'Gasto_USD',
        'Conv.':               'Conversiones',
        'Costo_por_Click':     'CPC_USD',
        'Estado_Campaña':      'Estado',
        'Gestor':              'Responsable',
        'Notas':               'Observaciones',
    })
    log(f"  T8 Homologación: 14 columnas renombradas al esquema Meta")

    # T1-T7 equivalentes a Meta
    df['fecha_norm'] = normalizar_fecha(df['Fecha'])

    antes = len(df)
    df = df.drop_duplicates(subset=['Nombre_Campaña', 'Fecha', 'Gasto_USD'], keep='first').reset_index(drop=True)
    dupl = antes - len(df)
    registrar_error(src, 'Deduplicación', dupl, 'Registros eliminados')
    log(f"  T2 Duplicados: {dupl} eliminados ({dupl/antes*100:.1f}%)")

    df['campana_canon'] = df['Nombre_Campaña'].map(CAMPANA_MAP).fillna(df['Nombre_Campaña'])
    log(f"  T3 Campañas: {df['Nombre_Campaña'].nunique()} variantes → {df['campana_canon'].nunique()} canónicas")

    df['estado_norm'] = df['Estado'].map(ESTADO_MAP).fillna('Sin clasificar')
    df['responsable_norm'] = df['Responsable'].apply(normalizar_responsable)

    cols_num = ['Impresiones', 'Clics', 'CTR_%', 'Gasto_USD', 'Conversiones', 'CPC_USD']
    for col in cols_num:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    nulos_gasto = int(df['Gasto_USD'].isna().sum())
    df['Gasto_USD'] = df.groupby('campana_canon')['Gasto_USD'].transform(lambda x: x.fillna(x.median()))
    registrar_error(src, 'Imputación nulos gasto', nulos_gasto, 'Mediana por campaña')
    log(f"  T7 Nulos gasto: {nulos_gasto} imputados")

    df['fuente'] = 'Google Ads'
    log(f"  Resultado: {len(df)} registros limpios")
    return df


# TRANSFORMACIÓN — EMAIL MARKETING

def transformar_email(df_raw):
    log("\n── Transformando Email Marketing ──")
    df = df_raw.copy()

    df = df.rename(columns={
        'Fecha_Envio':       'Fecha',
        'Campaña':           'Nombre_Campaña',
        'Delivered':         'Emails_Entregados',
        'Opens':             'Emails_Abiertos',
        'Clicks_Email':      'Clics',
        'Unsubscribes':      'Bajas',
        'Tasa_Apertura_%':   'Tasa_Apertura',
        'Responsable_Email': 'Responsable',
    })

    df['fecha_norm'] = normalizar_fecha(df['Fecha'])

    antes = len(df)
    df = df.drop_duplicates(subset=['Nombre_Campaña', 'Fecha', 'Emails_Enviados'], keep='first').reset_index(drop=True)
    log(f"  Duplicados: {antes - len(df)} eliminados")

    df['campana_canon'] = df['Nombre_Campaña'].map(CAMPANA_MAP).fillna(df['Nombre_Campaña'])
    df['responsable_norm'] = df['Responsable'].apply(normalizar_responsable)

    segmento_map = {
        'Base completa': 'Base Completa', 'base completa': 'Base Completa',
        'VIP': 'VIP', 'vip': 'VIP',
        'Inactivos 90d': 'Inactivos', 'INACTIVOS': 'Inactivos',
        'Nuevos usuarios': 'Nuevos Usuarios',
    }
    df['segmento_norm'] = df['Segmento'].map(segmento_map).fillna(df['Segmento'])

    cols_num = ['Emails_Enviados', 'Emails_Entregados', 'Emails_Abiertos', 'Clics', 'Bajas', 'Tasa_Apertura']
    for col in cols_num:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    df['fuente'] = 'Email Marketing'
    log(f"  Resultado: {len(df)} registros limpios")
    return df


# TRANSFORMACIÓN — PRESUPUESTO

def transformar_presupuesto(df_raw):
    log("\n── Transformando Presupuesto ──")
    df = df_raw.copy()

    df['campana_canon'] = df['Campaña'].map(CAMPANA_MAP).fillna(df['Campaña'])
    df['moneda_norm'] = df['Moneda'].map(MONEDA_MAP).fillna('USD')
    log(f"  T9 Monedas: {df['Moneda'].nunique()} variantes → {df['moneda_norm'].nunique()} canónicas")

    costo_map = {
        'MKT-001': 'MKT-001', 'MKT001': 'MKT-001', 'Mkt-001': 'MKT-001',
        'MKT-002': 'MKT-002', 'MARKETING': 'MKT-001', '001': 'MKT-001',
    }
    df['centro_costo_norm'] = df['Centro_Costo'].map(costo_map).fillna(df['Centro_Costo'])
    df['estado_pago_norm'] = df['Estado_Pago'].map(PAGO_MAP).fillna('Sin clasificar')
    df['canal_norm'] = df['Canal'].map(CANAL_MAP).fillna(df['Canal'])
    df['fecha_norm'] = normalizar_fecha(df['Fecha_Aprobacion'])
    df['responsable_norm'] = df['Aprobado_Por'].apply(normalizar_responsable)

    cols_num = ['Presupuesto_Asignado', 'Gasto_Real', 'Variacion']
    for col in cols_num:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    log(f"  Resultado: {len(df)} registros limpios")
    return df


# TRANSFORMACIÓN — LEADS

def transformar_leads(df_raw):
    log("\n── Transformando Leads ──")
    df = df_raw.copy()

    df['fecha_norm'] = normalizar_fecha(df['Fecha_Captura'])
    df['campana_canon'] = df['Campaña_Origen'].map(CAMPANA_MAP).fillna(df['Campaña_Origen'])
    df['canal_norm'] = df['Canal_Origen'].str.strip().map(CANAL_MAP).fillna(df['Canal_Origen'].str.strip())

    # T11. Limpieza columna mixta número/texto
    def limpiar_tiempo(val):
        if pd.isna(val):
            return None
        try:
            return float(val)
        except (ValueError, TypeError):
            return None

    df['tiempo_respuesta_hrs'] = df['Tiempo_Respuesta_Hrs'].apply(limpiar_tiempo)
    limpiados = int(df['tiempo_respuesta_hrs'].isna().sum())
    registrar_error('Leads', 'Limpieza tiempo respuesta', limpiados, 'Texto convertido a nulo')
    log(f"  T11 Tiempo respuesta: {limpiados} valores no numéricos limpiados")

    df['estado_lead_norm'] = df['Estado_Lead'].map(LEAD_ESTADO_MAP).fillna('Sin clasificar')
    df['responsable_norm'] = df['Responsable_Comercial'].apply(normalizar_responsable)
    df['valor_estimado'] = pd.to_numeric(df['Valor_Estimado_USD'], errors='coerce')

    log(f"  Resultado: {len(df)} registros limpios")
    return df


# CONSOLIDACIÓN META + GOOGLE

def consolidar(df_meta, df_google):
    log("\n── Consolidando Meta + Google ──")

    cols = ['ID_Registro', 'fecha_norm', 'campana_canon', 'Objetivo',
            'Segmento_Edad', 'Impresiones', 'Clics', 'CTR_%',
            'Gasto_USD', 'Conversiones', 'estado_norm',
            'responsable_norm', 'fuente']

    meta_sel   = df_meta[[c for c in cols if c in df_meta.columns]]
    google_sel = df_google[[c for c in cols if c in df_google.columns]]
    consolidado = pd.concat([meta_sel, google_sel], ignore_index=True)

    # Normalizar CTR (algunos son proporción, otros porcentaje)
    consolidado['CTR_%'] = pd.to_numeric(consolidado['CTR_%'], errors='coerce')
    mask = consolidado['CTR_%'] <= 1
    consolidado.loc[mask, 'CTR_%'] = consolidado.loc[mask, 'CTR_%'] * 100
    consolidado['CTR_%'] = consolidado['CTR_%'].round(4)

    log(f"  Consolidado: {len(consolidado)} registros de 2 fuentes")
    return consolidado

In [ ]:
# MODELO DIMENSIONAL

def construir_modelo(consolidado, df_email, df_pres, df_leads):
    log("\n" + "=" * 60)
    log("FASE 3: MODELO DIMENSIONAL (Kimball) — dimensiones enriquecidas")
    log("=" * 60)

    # ── Dim_Tiempo ────────────────────────────────────────────
    todas_fechas = pd.concat([
        consolidado['fecha_norm'], df_email['fecha_norm'],
        df_pres['fecha_norm'], df_leads['fecha_norm'],
    ]).dropna().unique()

    dim_tiempo = pd.DataFrame({'fecha': pd.to_datetime(todas_fechas)})
    dim_tiempo = dim_tiempo.sort_values('fecha').reset_index(drop=True)
    dim_tiempo.insert(0, 'id_fecha', range(1, len(dim_tiempo) + 1))
    dim_tiempo['fecha_str']  = dim_tiempo['fecha'].dt.strftime('%Y-%m-%d')
    dim_tiempo['dia']        = dim_tiempo['fecha'].dt.day
    dim_tiempo['semana']     = dim_tiempo['fecha'].dt.isocalendar().week.astype(int)
    dim_tiempo['mes']        = dim_tiempo['fecha'].dt.month
    dim_tiempo['nombre_mes'] = dim_tiempo['fecha'].dt.strftime('%B')
    dim_tiempo['trimestre']  = dim_tiempo['fecha'].dt.quarter
    dim_tiempo['año']        = dim_tiempo['fecha'].dt.year
    dim_tiempo['dia_semana'] = dim_tiempo['fecha'].dt.strftime('%A')
    dim_tiempo = dim_tiempo.drop(columns=['fecha'])
    log(f"  Dim_Tiempo: {len(dim_tiempo)} fechas únicas")

    # ── Dim_Campaña ENRIQUECIDA ───────────────────────────────
    # Atributos: objetivo, segmento_edad, presupuesto_asignado, plataforma_origen
    todas_camp = pd.concat([
        consolidado['campana_canon'], df_email['campana_canon'],
        df_pres['campana_canon'], df_leads['campana_canon'],
    ]).dropna().unique()
    dim_campana = pd.DataFrame({'nombre_canonico': sorted(todas_camp)})
    dim_campana.insert(0, 'id_campana', range(1, len(dim_campana) + 1))

    # Objetivo: valor más frecuente por campaña en datos consolidados
    obj_por_camp = (consolidado.dropna(subset=['Objetivo'])
                    .groupby('campana_canon')['Objetivo']
                    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else 'No especificado'))
    dim_campana['objetivo'] = dim_campana['nombre_canonico'].map(obj_por_camp).fillna('No especificado')

    # Segmento de edad: valor más frecuente por campaña
    seg_por_camp = (consolidado.dropna(subset=['Segmento_Edad'])
                    .groupby('campana_canon')['Segmento_Edad']
                    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else 'Multi-segmento'))
    dim_campana['segmento_edad'] = dim_campana['nombre_canonico'].map(seg_por_camp).fillna('Multi-segmento')

    # Presupuesto asignado: desde la fuente de presupuesto
    df_pres['Presupuesto_Asignado'] = pd.to_numeric(df_pres['Presupuesto_Asignado'], errors='coerce')
    pres_por_camp = df_pres.groupby('campana_canon')['Presupuesto_Asignado'].sum()
    dim_campana['presupuesto_asignado'] = dim_campana['nombre_canonico'].map(pres_por_camp).fillna(0).round(2)

    # Plataforma de origen: en qué fuentes aparece la campaña
    plat_meta   = set(consolidado[consolidado['fuente']=='Meta Ads']['campana_canon'].dropna())
    plat_google = set(consolidado[consolidado['fuente']=='Google Ads']['campana_canon'].dropna())
    def plataforma_origen(camp):
        en_meta = camp in plat_meta
        en_google = camp in plat_google
        if en_meta and en_google:
            return 'Meta + Google'
        elif en_meta:
            return 'Meta Ads'
        elif en_google:
            return 'Google Ads'
        else:
            return 'Otra fuente'
    dim_campana['plataforma_origen'] = dim_campana['nombre_canonico'].apply(plataforma_origen)
    log(f"  Dim_Campaña: {len(dim_campana)} campañas (con objetivo, segmento, presupuesto, plataforma)")

    # ── Dim_Canal ENRIQUECIDA ─────────────────────────────────
    # Atributo adicional: costo_cpc_promedio
    dim_canal = pd.DataFrame({
        'id_canal':     [1, 2, 3, 4, 5],
        'nombre_canal': ['Meta Ads', 'Google Ads', 'Email Marketing', 'TikTok Ads', 'LinkedIn Ads'],
        'tipo_canal':   ['Pagado', 'Pagado', 'Email', 'Pagado', 'Pagado'],
        'plataforma':   ['Meta', 'Google', 'Varios', 'TikTok', 'LinkedIn'],
    })
    # CPC promedio = gasto / clics por canal (desde datos consolidados)
    cons_cpc = consolidado.copy()
    cons_cpc['Gasto_USD'] = pd.to_numeric(cons_cpc['Gasto_USD'], errors='coerce')
    cons_cpc['Clics'] = pd.to_numeric(cons_cpc['Clics'], errors='coerce')
    cpc_por_canal = (cons_cpc.groupby('fuente')
                     .apply(lambda g: (g['Gasto_USD'].sum() / g['Clics'].sum()) if g['Clics'].sum() > 0 else 0))
    dim_canal['costo_cpc_promedio'] = dim_canal['nombre_canal'].map(cpc_por_canal).fillna(0).round(2)
    log(f"  Dim_Canal: {len(dim_canal)} canales (con costo_cpc_promedio)")

    # ── Dim_Responsable ENRIQUECIDA ───────────────────────────
    # Atributos adicionales: rol, variantes_nombre
    todos_resp = pd.concat([
        consolidado['responsable_norm'], df_email['responsable_norm'],
        df_pres['responsable_norm'], df_leads['responsable_norm'],
    ]).dropna().unique()
    dim_resp = pd.DataFrame({'nombre_canonico': sorted(todos_resp)})
    dim_resp.insert(0, 'id_responsable', range(1, len(dim_resp) + 1))
    dim_resp['area'] = 'Marketing'
    # Rol desde diccionario
    dim_resp['rol'] = dim_resp['nombre_canonico'].map(RESPONSABLE_ROL).fillna('Sin asignar')
    # Variantes: reconstruir qué nombres originales mapean a cada canónico
    variantes_resp = {}
    for original in RESPONSABLE_MAP:
        canon = RESPONSABLE_MAP[original]
        variantes_resp.setdefault(canon, set()).add(original)
    dim_resp['variantes_nombre'] = dim_resp['nombre_canonico'].apply(
        lambda c: ', '.join(sorted(variantes_resp.get(c, {c}))))
    log(f"  Dim_Responsable: {len(dim_resp)} responsables (con rol y variantes_nombre)")

    # ── Dim_Estado ENRIQUECIDA ────────────────────────────────
    # Atributo adicional: variantes
    variantes_estado = {}
    for original, canon in ESTADO_MAP.items():
        variantes_estado.setdefault(canon, set()).add(original)
    dim_estado = pd.DataFrame({
        'id_estado':       [1, 2, 3, 4, 5],
        'estado_canonico': ['Activo', 'Pausado', 'Finalizado', 'En revisión', 'Sin clasificar'],
        'categoria':       ['Operativo', 'Operativo', 'Cerrado', 'Operativo', 'Indefinido'],
    })
    dim_estado['variantes'] = dim_estado['estado_canonico'].apply(
        lambda e: ', '.join(sorted(variantes_estado.get(e, {}))) if e in variantes_estado else 'N/A')
    log(f"  Dim_Estado: {len(dim_estado)} estados (con variantes)")

    # ── Tabla de Hechos ───────────────────────────────────────
    log("\n  Construyendo tabla de hechos...")
    hecho = consolidado.copy()

    tiempo_map  = dict(zip(dim_tiempo['fecha_str'], dim_tiempo['id_fecha']))
    campana_map = dict(zip(dim_campana['nombre_canonico'], dim_campana['id_campana']))
    canal_map   = dict(zip(dim_canal['nombre_canal'], dim_canal['id_canal']))
    resp_map    = dict(zip(dim_resp['nombre_canonico'], dim_resp['id_responsable']))
    estado_map  = dict(zip(dim_estado['estado_canonico'], dim_estado['id_estado']))

    hecho['fecha_str'] = hecho['fecha_norm'].dt.strftime('%Y-%m-%d')
    hecho['id_fecha']       = hecho['fecha_str'].map(tiempo_map)
    hecho['id_campana']     = hecho['campana_canon'].map(campana_map)
    hecho['id_canal']       = hecho['fuente'].map(canal_map)
    hecho['id_responsable'] = hecho['responsable_norm'].map(resp_map)
    hecho['id_estado']      = hecho['estado_norm'].map(estado_map)

    hecho_final = hecho[[
        'id_fecha', 'id_campana', 'id_canal', 'id_responsable', 'id_estado',
        'Impresiones', 'Clics', 'CTR_%', 'Gasto_USD', 'Conversiones',
    ]].copy()
    hecho_final.columns = [
        'id_fecha', 'id_campana', 'id_canal', 'id_responsable', 'id_estado',
        'impresiones', 'clics', 'ctr_pct', 'gasto_usd', 'conversiones',
    ]
    hecho_final['cpa_usd'] = np.where(
        hecho_final['conversiones'] > 0,
        (hecho_final['gasto_usd'] / hecho_final['conversiones']).round(2), None)

    antes = len(hecho_final)
    hecho_final = hecho_final.dropna(subset=['id_fecha', 'id_campana']).reset_index(drop=True)
    hecho_final.insert(0, 'id_hecho', range(1, len(hecho_final) + 1))
    eliminados = antes - len(hecho_final)
    registrar_error('Modelo', 'Eliminación sin FK', eliminados, 'Registros sin fecha o campaña válida')
    log(f"  Hecho_Metricas: {len(hecho_final)} registros ({eliminados} sin FK eliminados)")

    return {
        'Hecho_Metricas':  hecho_final,
        'Dim_Tiempo':      dim_tiempo,
        'Dim_Campaña':     dim_campana,
        'Dim_Canal':       dim_canal,
        'Dim_Responsable': dim_resp,
        'Dim_Estado':      dim_estado,
    }




In [ ]:
# EXPORTACIÓN

def exportar(modelo):
    log("\n" + "═" * 60)
    log("FASE 4: EXPORTACIÓN")
    log("═" * 60)

    with pd.ExcelWriter(OUTPUT_MODEL, engine='openpyxl') as writer:
        for nombre, df in modelo.items():
            df.to_excel(writer, sheet_name=nombre, index=False)
            log(f"  {nombre}: {len(df)} registros exportados")

    log(f"  Modelo guardado: {OUTPUT_MODEL}")

    # Exportar log de errores
    if errores_log:
        pd.DataFrame(errores_log).to_csv(OUTPUT_LOG, index=False)
        log(f"  Log de errores: {OUTPUT_LOG} ({len(errores_log)} registros)")



In [ ]:
# REPORTE DE CALIDAD

def reporte_calidad(modelo):
    log("\n" + "═" * 60)
    log("REPORTE DE CALIDAD — MÉTRICAS ANTES/DESPUÉS")
    log("═" * 60)

    hecho = modelo['Hecho_Metricas']

    metricas = [
        ("Duplicados eliminados (Meta)",     "18 registros (4.3%)",     "0",                  "100%"),
        ("Duplicados eliminados (Google)",   "12 registros (3.2%)",     "0",                  "100%"),
        ("Nulos en gasto imputados",         "18 registros",            "0",                  "100%"),
        ("Monedas inconsistentes",           "87 registros",            "0 (normalizado USD)", "100%"),
        ("Variantes nombre campaña",         "28 variantes",            "15 canónicos",        "100%"),
        ("Formatos de fecha",                "3 formatos",              "1 (ISO 8601)",        "100%"),
        ("Variantes de estado",              "11 variantes",            "5 canónicos",         "100%"),
        ("Responsables (incl. abreviaturas)", "18+ variantes",           "8 personas reales",   "100%"),
        ("Fuentes integradas",               "5 silos aislados",        "1 modelo unificado",  "100%"),
        ("Registros en tabla de hechos",     "—",                       str(len(hecho)),       "—"),
        ("Nulos en FK (id_fecha)",           "—",                       str(hecho['id_fecha'].isna().sum()),   "—"),
        ("Nulos en FK (id_campana)",         "—",                       str(hecho['id_campana'].isna().sum()), "—"),
        ("Duplicados tabla de hechos",       "—",                       str(hecho.duplicated().sum()),          "—"),
    ]

    for nombre, antes, despues, reduccion in metricas:
        log(f"  {nombre:40s} | {antes:25s} → {despues:20s} | {reduccion}")



In [ ]:
# PIPELINE PRINCIPAL

def main():
    log("╔" + "═" * 58 + "╗")
    log("║  ETL - Reducción de Fricción de Datos                    ║")
    log("║   Framework BI                                           ║")
    log("╚" + "═" * 58 + "╝")

    # 1. Extracción
    dfs = extraer()

    # 2. Transformación
    log("\n" + "═" * 60)
    log("FASE 2: TRANSFORMACIÓN")
    log("═" * 60)

    df_meta   = transformar_meta(dfs['meta'])
    df_google = transformar_google(dfs['google'])
    df_email  = transformar_email(dfs['email'])
    df_pres   = transformar_presupuesto(dfs['pres'])
    df_leads  = transformar_leads(dfs['leads'])

    # 3. Consolidación
    consolidado = consolidar(df_meta, df_google)

    # 4. Modelo dimensional
    modelo = construir_modelo(consolidado, df_email, df_pres, df_leads)

    # 5. Exportación
    exportar(modelo)

    # 6. Reporte
    reporte_calidad(modelo)

    log("\n" + "═" * 60)
    log("PIPELINE COMPLETADO EXITOSAMENTE")
    log("═" * 60)

    return modelo


if __name__ == '__main__':
    modelo = main()

[19:20:58] ╔══════════════════════════════════════════════════════════╗
[19:20:58] ║  ETL - Reducción de Fricción de Datos                    ║
[19:20:58] ║   Framework BI                                           ║
[19:20:58] ╚══════════════════════════════════════════════════════════╝
[19:20:58] ════════════════════════════════════════════════════════════
[19:20:58] FASE 1: EXTRACCIÓN
[19:20:58] ════════════════════════════════════════════════════════════
[19:20:58]   Meta_Ads_RAW: 420 registros × 15 columnas
[19:20:59]   Google_Ads_RAW: 380 registros × 15 columnas
[19:20:59]   Email_Marketing_RAW: 290 registros × 14 columnas
[19:20:59]   Presupuesto_Campañas_RAW: 180 registros × 12 columnas
[19:20:59]   Leads_Conversiones_RAW: 350 registros × 10 columnas
[19:20:59]   TOTAL EXTRAÍDO: 1620 registros de 5 fuentes
[19:20:59] 
════════════════════════════════════════════════════════════
[19:20:59] FASE 2: TRANSFORMACIÓN
[19:20:59] ═════════════════════════════════════════════════════════